In [ ]:
pip install pandas numpy matplotlib yfinance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
plt.style.use("seaborn-v0_8")
from datetime import date
pd.options.display.float_format = '{:.2f}'.format


class StockValuator:

    def __init__(self, tickers: list, start: str = '2022-01-01', end: str = None):
        self.tickers = tickers
        self.start = start
        self.end = end if end else date.today().strftime('%Y-%m-%d')
        self.stocks = None
        self.normalized = None
        self.returns = None
        self.summary_df = None
        self.errors = []

    def financials_load(self):
        raw = yf.download(tickers=self.tickers, start=self.start, end=self.end, progress=False).Close

        missing = raw.columns[raw.isna().all()].tolist()
        if missing:
            self.errors.append(f"No data returned for: {missing}. Dropped.")
            raw = raw.drop(columns=missing)

        high_missing = raw.columns[raw.isna().mean() > 0.20].tolist()
        if high_missing:
            self.errors.append(f"Tickers with >20% missing data dropped: {high_missing}")
            raw = raw.drop(columns=high_missing)

        self.stocks = raw.dropna()
        self.tickers = self.stocks.columns.tolist()

    def get_current_shares_outstanding(self):
        shares_out = {}
        for ticker in self.tickers:
            try:
                stock = yf.Ticker(ticker)
                shares = stock.info.get("sharesOutstanding")
                if shares is None:
                    self.errors.append(f"{ticker}: sharesOutstanding unavailable, excluded from VWI.")
                shares_out[ticker] = shares
            except Exception as e:
                self.errors.append(f"{ticker}: {e}")
                shares_out[ticker] = None
        return pd.Series(shares_out, name="SharesOutstanding")

    def calculations(self):
        self.normalized = self.stocks.div(self.stocks.iloc[0]) * 100
        self.normalized['PWI'] = (
            self.stocks.sum(axis=1)
            .div(self.stocks.sum(axis=1).iloc[0]) * 100
        )

        self.returns = self.stocks.pct_change().dropna()
        self.returns['mean'] = self.returns.mean(axis=1)

        self.normalized['EWI'] = 100.0
        self.normalized.iloc[1:, self.normalized.columns.get_loc('EWI')] = (
            self.returns['mean'].add(1).cumprod().mul(100)
        )

        shares_outstanding = self.get_current_shares_outstanding()
        valid = shares_outstanding.dropna()

        if valid.empty:
            self.errors.append("VWI skipped: no shares outstanding data available for any ticker.")
        else:
            stocks_vwi = self.stocks[valid.index]
            mcap = stocks_vwi.mul(valid, axis='columns')
            weights_vwi = mcap.div(mcap.sum(axis=1), axis='index')
            stock_returns = self.returns[valid.index]
            self.normalized['VWI'] = (
                stock_returns
                .mul(weights_vwi.shift().dropna())
                .sum(axis=1)
                .add(1).cumprod().mul(100)
            )

    def summary(self):
        self.summary_df = (
            self.normalized.pct_change().dropna()
            .agg(['mean', 'std']).T
        )
        self.summary_df['mean'] = self.summary_df['mean'].mul(252)
        self.summary_df['std'] = self.summary_df['std'].mul(np.sqrt(252))
        self.summary_df['sharpe'] = self.summary_df['mean'].div(self.summary_df['std'])

        indices = [c for c in ['PWI', 'EWI', 'VWI'] if c in self.summary_df.index]
        return self.summary_df[self.summary_df.index.isin(indices)]

    def graphs(self, save_path: str = None):
        highlight = [c for c in ['PWI', 'EWI', 'VWI'] if c in self.summary_df.index]

        default_data = self.summary_df[~self.summary_df.index.isin(highlight)]
        highlight_data = self.summary_df[self.summary_df.index.isin(highlight)]

        fig, ax = plt.subplots(figsize=(15, 8))
        ax.scatter(default_data['std'], default_data['mean'], s=50, label='Stocks')
        ax.scatter(highlight_data['std'], highlight_data['mean'], s=70, color='red', label='Indices')

        for i in self.summary_df.index:
            ax.annotate(
                i,
                xy=(
                    self.summary_df.loc[i, 'std'] + 0.002,
                    self.summary_df.loc[i, 'mean'] + 0.002,
                ),
                fontsize=15,
            )

        ax.set_xlabel('ann. Risk (std)', fontsize=15)
        ax.set_ylabel('ann. Return', fontsize=15)
        ax.set_title('Risk / Return', fontsize=20)
        ax.legend(fontsize=13)
        plt.tight_layout()

        if save_path:
            fig.savefig(save_path, dpi=150, bbox_inches='tight')

        plt.show()
        return fig

    def error_summary(self):
        if not self.errors:
            print("No errors or warnings.")
        else:
            print("\n=== Error / Warning Summary ===")
            for i, e in enumerate(self.errors, 1):
                print(f"  {i}. {e}")

    def main(self):
        self.financials_load()
        self.calculations()
        index_summary = self.summary()
        self.error_summary()
        print("\n=== Index Risk / Return Summary ===")
        print(index_summary)
        self.graphs(save_path='outputs/risk_return.png')
        return index_summary

#Entry Point
if __name__ == '__main__':
    tickers = ['AAPL', 'MSFT', 'AMZN', 'NVDA', 'GOOGL', 'META',
    'LLY', 'AVGO', 'TSLA', 'JPM', 'UNH', 'V', 'XOM', 'MA', 'COST',
    'PG', 'JNJ', 'HD', 'MRK', 'CVX', 'PEP', 'KO'] #Insert tickers here
    sv = StockValuator(tickers)
    sv.main()